# Session 29 — LangChain Fundamentals & Components
### Hands-on Notebook · Computer Vision & AI series

This notebook is the practical half of Session 29. It mirrors the slides and lets you *run* every core LangChain building block on your own machine — **free, locally, no paid API keys** — using a small Hugging Face model.

**Part 1 — Fundamentals:** Models · Prompts · Output Parsers · Chains (LCEL)
**Part 2 — Components:** Memory · Document Loaders · Text Splitters
**Mini project:** wire the pieces into a tiny retrieval-augmented Q&A.

> Aligned to **LangChain v1.0** (late 2025). New code uses the **LCEL pipe** (`prompt | model | parser`); the older `LLMChain`/`SequentialChain` classes now live in the separate `langchain-classic` package.

**How to use it:** run the cells top to bottom. Read the short note above each code cell, run it, then tweak the inputs and re-run to see how the pieces compose.


## 0 · Setup

We install the LangChain packages plus Hugging Face's `transformers` and `torch`. LangChain is deliberately modular, so each capability is its own small package:

| Package | Gives us |
|---|---|
| `langchain-core` | prompts, output parsers, the Runnable/LCEL interface |
| `langchain-huggingface` | run Hugging Face models through LangChain |
| `langchain-community` | document loaders |
| `langchain-text-splitters` | chunking utilities |

Run the install cell once (it may take a minute). If you're on Colab, restart is not required.

In [10]:
# Run once. Quiet install of everything this notebook needs.
%pip install -q -U langchain-core langchain-huggingface langchain-community \
    langchain-text-splitters transformers
print("Setup complete.")

Setup complete.


In [11]:
# Quick import check — confirms the packages above are available.
import warnings; warnings.filterwarnings("ignore")   # keep teaching output clean
import langchain_core
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
print("langchain-core version:", langchain_core.__version__)
print("Imports OK — ready to go.")

langchain-core version: 1.4.9
Imports OK — ready to go.


---
# Part 1 — Fundamentals

The anatomy of a single LLM call: **Models → Prompts → Output Parsers → Chains**.

## 1 · Models — the reasoning engine

A **model** is the component that actually generates text. LangChain wraps every provider behind the *same* interface, so the rest of your code doesn't change when you swap the model.

We'll use **`HuggingFacePipeline`**, which downloads a small open model and runs it **locally**. Our default is `google/flan-t5-small` — tiny, fast, and CPU-friendly, perfect for teaching. You can later swap in `flan-t5-base` / `flan-t5-large` for better answers, or point at a hosted model with `HuggingFaceEndpoint` — the code below it stays identical.

*(flan-t5 is an instruction-tuned text-to-text model, so the task is `"text2text-generation"`.)*

In [8]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "microsoft/Phi-3-mini-4k-instruct",
    model_provider="huggingface",
    temperature=0.7,
    max_tokens=1024,
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
print(model.invoke("Explain what a neural network is in one sentence."))

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


content="<|user|>\nExplain what a neural network is in one sentence.<|end|>\n<|assistant|>\nA neural network is a computational model inspired by the human brain's structure, consisting of interconnected nodes that process and analyze complex patterns and data to perform tasks like classification, prediction, and decision-making." additional_kwargs={} response_metadata={} id='lc_run--019f8ac7-a656-75e2-a5d1-f81101782c2e-0' tool_calls=[] invalid_tool_calls=[]


**What just happened?** `llm` is a *Runnable* — every LangChain block shares three methods: `.invoke()` (one input), `.stream()` (token by token), and `.batch()` (many inputs at once). Keep that in mind; it's the key idea that makes chains possible.

In [12]:
# .batch() runs several prompts efficiently in one call.
answers = model.batch([
    "Name one use of computer vision.",
    "What does 'RGB' stand for?",
])
for a in answers:
    print("-", a)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- content='<|user|>\nName one use of computer vision.<|end|>\n<|assistant|>\nOne use of computer vision is facial recognition technology. This technology uses computer vision algorithms to identify and verify individuals based on their facial features. It is used in various applications, such as unlocking smartphones, security systems, and identifying suspects in law enforcement.' additional_kwargs={} response_metadata={} id='lc_run--019f8ae9-227a-7fc1-a910-074ad4bf4c45-0' tool_calls=[] invalid_tool_calls=[]
- content="<|user|>\nWhat does 'RGB' stand for?<|end|>\n<|assistant|>\nRGB stands for Red, Green, and Blue. It is a color model used in various devices like television screens, computer monitors, and cameras to create a wide spectrum of colors. Each color in the model is represented by a numerical value, where red, green, and blue are mixed in different proportions to form millions of colors." additional_kwargs={} response_metadata={} id='lc_run--019f8ae9-227d-72d3-89a5-2298085748e

## 2 · Prompts — reusable, parameterized instructions

Hard-coding the exact wording of every request is brittle. A **prompt template** is a recipe with blanks (`{like_this}`). You write it once, then fill the blanks at runtime — so the same template serves any input.

- **`PromptTemplate`** — a single templated string.
- **`ChatPromptTemplate`** — separate **system** (the role/rules) and **human** (the request) messages.

In [13]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# (a) Simple string template
simple = PromptTemplate.from_template(
    "Explain {topic} to a beginner in 2 short sentences."
)
print("Filled string prompt:\n", simple.format(topic="image embeddings"))

# (b) Chat template with roles
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly computer-vision tutor. Keep answers simple."),
    ("human",  "Explain {topic} to a beginner."),
])
print("\nFilled chat prompt:")
for m in chat_prompt.invoke({"topic": "convolution"}).to_messages():
    print(f"  [{m.type}] {m.content}")

Filled string prompt:
 Explain image embeddings to a beginner in 2 short sentences.

Filled chat prompt:
  [system] You are a friendly computer-vision tutor. Keep answers simple.
  [human] Explain convolution to a beginner.


Notice we only changed the value of `{topic}`. The template is **reusable** — that reusability is exactly what lets us drop a prompt into a chain and feed it different data every call.

## 3 · Output Parsers — from raw text to usable data

Models return plain text. A **parser** is the final step that turns that text into something your program can *use*.

- **`StrOutputParser`** → pulls out the clean string.
- **`JsonOutputParser`** → parses a JSON reply into a Python `dict`.
- **`PydanticOutputParser`** / **`with_structured_output()`** → validate against a schema (a typed object).

In [14]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# StrOutputParser: model output -> plain string (a no-op cleanup here, but
# essential once you're inside a chain and want just the text).
str_parser = StrOutputParser()
print("String parser:", str_parser.invoke("A tidy answer."))

# JsonOutputParser: turn a JSON *string* into a Python dict you can index.
json_parser = JsonOutputParser()
parsed = json_parser.parse('{"object": "cat", "confidence": 0.92}')
print("Parsed dict:", parsed, "| confidence:", parsed["confidence"])

String parser: A tidy answer.
Parsed dict: {'object': 'cat', 'confidence': 0.92} | confidence: 0.92


> **Why parse at all?** Without a parser you're left string-wrangling model output by hand. With one, the output drops straight into the rest of your code as a value you can trust.
>
> For strict, typed results define a schema with **Pydantic** and use `model.with_structured_output(YourSchema)` — great for extraction tasks. (Small models like flan-t5-small aren't reliable at emitting JSON on demand, which is exactly why we parsed a fixed string above.)

## 4 · Chains — composing steps with LCEL

Now we connect the pieces with the pipe operator **`|`**. This is **LangChain Expression Language (LCEL)**. Because every block is a Runnable, the whole chain is *also* a Runnable — so it too has `.invoke()`, `.stream()`, and `.batch()`.

```
chain = prompt | model | parser
```

Data flows left → right: the prompt fills its blanks, the model generates, the parser cleans up.

In [15]:
# Build the classic three-step chain with LCEL.
chain = simple | model | StrOutputParser()   # prompt | model | parser

# Run it — we only pass the template variable.
print(chain.invoke({"topic": "edge detection"}))

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|>
Explain edge detection to a beginner in 2 short sentences.<|end|>
<|assistant|>
Edge detection is a process in digital image processing that identifies points in a digital image where the brightness changes sharply or has discontinuities. It's like finding the outlines of objects within an image, which helps in object recognition and scene understanding.


In [16]:
# Because the chain is itself a Runnable, .batch() works on the whole pipeline.
for out in chain.batch([{"topic": "pixels"}, {"topic": "a dataset"}]):
    print("-", out)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- <|user|>
Explain pixels to a beginner in 2 short sentences.<|end|>
<|assistant|>
Pixels are the tiny dots that make up the images on a screen. The more pixels there are, the sharper and more detailed the image appears.
- <|user|>
Explain a dataset to a beginner in 2 short sentences.<|end|>
<|assistant|>
A dataset is a collection of related data points, often organized in rows and columns, which can be analyzed and interpreted to extract useful information for various purposes. Think of it as a digital spreadsheet where each cell holds a specific piece of data.


That's the entire foundation of LangChain: **uniform Runnables piped together.** Everything in Part 2 is just adding or swapping a block.

---
# Part 2 — Components

The pieces that turn a single call into a **stateful, data-aware application**: **Memory → Document Loaders → Text Splitters** (then a mini end-to-end).

## 5 · Memory — remembering the conversation

By default each call is **stateless** — the model forgets everything the moment it answers. **Memory** feeds the earlier turns back in so a conversation can build on itself.

We'll attach memory to our chain with **`RunnableWithMessageHistory`**. The idea:
1. A **`MessagesPlaceholder`** in the prompt reserves a slot for past messages.
2. A **history store** keeps the running transcript, looked up by a **`session_id`** (one id = one conversation, so different users/threads stay separate).

*(In production LangChain v1.0 apps, this same idea is handled by a **checkpointer** keyed on a `thread_id`, plus middleware to trim or summarize long histories. `RunnableWithMessageHistory` is the simplest way to see the concept — you may notice a deprecation notice steering you toward LangGraph persistence for real apps; the underlying idea is identical.)*

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# Prompt with a slot for prior messages ("history") and the new input.
conv_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation so far."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
conv_chain = conv_prompt | model | StrOutputParser()

# A tiny in-memory store: one InMemoryChatMessageHistory per session_id.
_store = {}
def get_history(session_id: str):
    if session_id not in _store:
        _store[session_id] = InMemoryChatMessageHistory()
    return _store[session_id]

chat = RunnableWithMessageHistory(
    conv_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)
print("Memory-enabled chain ready.")

Memory-enabled chain ready.


In [18]:
# Same conversation => same session_id. Watch the history accumulate.
cfg = {"configurable": {"session_id": "student-001"}}

print("Turn 1:", chat.invoke({"input": "My name is Bob and I study computer vision."}, config=cfg))
print("Turn 2:", chat.invoke({"input": "In one word, what field do I study?"}, config=cfg))

# Peek at what memory is holding for this session:
print("\nStored transcript:")
for m in get_history("student-001").messages:
    print(f"  [{m.type}] {m.content}")

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 1: <|system|>
You are a helpful assistant. Use the conversation so far.<|end|>
<|user|>
My name is Bob and I study computer vision.<|end|>
<|assistant|>
Hello! I am Phi. Your interest in computer vision is quite fascinating. This field combines elements of engineering and computer science to enable computers to gain high-level understanding from digital images or videos. What aspect of computer vision are you particularly interested in?
Turn 2: <|system|>
You are a helpful assistant. Use the conversation so far.<|end|>
<|user|>
My name is Bob and I study computer vision.<|end|>
<|assistant|>
<|system|>
You are a helpful assistant. Use the conversation so far.<|end|>
<|user|>
My name is Bob and I study computer vision.<|end|>
<|assistant|>
Hello! I am Phi. Your interest in computer vision is quite fascinating. This field combines elements of engineering and computer science to enable computers to gain high-level understanding from digital images or videos. What aspect of computer v

The second question never repeats the field — the model can answer only because the first turn was replayed from memory. Switch to a different `session_id` and that context is gone, exactly as you'd want for separate users.

*Note:* a tiny model may not always answer perfectly; the point here is the **mechanism** — how history is stored, keyed, and fed back in.

## 6 · Document Loaders — bring your own data in

To answer questions about *your* content (PDFs, notes, web pages), you first load it. **Loaders** read a source and return a list of standardized **`Document`** objects, each with:

- **`page_content`** — the text, and
- **`metadata`** — where it came from (source, page number, …), which flows through the pipeline so answers can cite sources.

We'll write a small text file and load it with **`TextLoader`**. Other common loaders: `PyPDFLoader` (PDF), `CSVLoader` (CSV), `WebBaseLoader` (web pages).

In [23]:
# Create a small knowledge file to load.
notes = '''Computer vision lets computers interpret images and video.
A convolutional neural network (CNN) is the classic model for image tasks.
Convolutions detect local patterns like edges, corners, and textures.
Pooling layers shrink feature maps so the network is faster and more robust.
Image embeddings turn a picture into a vector so similar images sit close together.
LangChain helps connect these ideas to large language models.'''

with open("cv_notes.txt", "w") as f:
    f.write(notes)

from langchain_community.document_loaders import TextLoader

docs = TextLoader("cv_notes.txt").load()
print("Loaded", len(docs), "document(s)")
print("metadata:", docs[0].metadata)
print("first 80 chars:", docs[0].page_content[:80], "...")

Loaded 1 document(s)
metadata: {'source': 'cv_notes.txt'}
first 80 chars: Computer vision lets computers interpret images and video.
A convolutional neura ...


Whatever the original format, downstream steps only ever see `page_content` + `metadata`. That uniformity is what makes the whole pipeline modular — you can change the *source* without changing anything after it.

## 7 · Text Splitters — chunk it for retrieval

A loaded document is often **too big** to fit in a prompt or an embedding. **Text splitters** cut it into smaller **chunks**. Two dials control this:

- **`chunk_size`** — the maximum characters per chunk.
- **`chunk_overlap`** — characters shared between neighbouring chunks, so a sentence isn't lost when it straddles a cut.

**`RecursiveCharacterTextSplitter`** is the recommended default: it tries to split on paragraphs, then lines, then words — keeping related text together.

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,     # small values so we can SEE the chunks
    chunk_overlap=20,   # neighbours share ~20 chars for context
)
chunks = splitter.split_documents(docs)

print(f"Split 1 document into {len(chunks)} chunks:\n")
for i, c in enumerate(chunks):
    print(f"chunk {i} ({len(c.page_content)} chars): {c.page_content!r}\n")

Split 1 document into 6 chunks:

chunk 0 (58 chars): 'Computer vision lets computers interpret images and video.'

chunk 1 (74 chars): 'A convolutional neural network (CNN) is the classic model for image tasks.'

chunk 2 (69 chars): 'Convolutions detect local patterns like edges, corners, and textures.'

chunk 3 (76 chars): 'Pooling layers shrink feature maps so the network is faster and more robust.'

chunk 4 (83 chars): 'Image embeddings turn a picture into a vector so similar images sit close together.'

chunk 5 (61 chars): 'LangChain helps connect these ideas to large language models.'



Try changing `chunk_size` and `chunk_overlap` and re-running. Bigger chunks = more context per piece but fewer, coarser matches; smaller chunks = sharper matches but more of them. Tuning this is one of the main quality levers in retrieval systems.

## 8 · Mini project — put it all together (tiny RAG)

Let's chain everything into a miniature **Retrieval-Augmented Generation** flow:

1. **Retrieve** the chunk most relevant to a question (here, a simple word-overlap score — in a real app you'd use embeddings + a vector store).
2. **Stuff** that chunk into a prompt as context.
3. **Generate** a grounded answer with our LCEL chain.

This is the same shape as production RAG — just with the simplest possible retriever so it runs instantly and offline.

In [26]:
# 1) A very small "retriever": score chunks by shared words with the question.
def retrieve(question, chunks, k=1):
    q_words = set(question.lower().split())
    scored = sorted(
        chunks,
        key=lambda c: len(q_words & set(c.page_content.lower().split())),
        reverse=True,
    )
    return scored[:k]

# 2) A prompt that grounds the answer in the retrieved context.
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the context. If unsure, say you don't know."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
rag_chain = rag_prompt | model | StrOutputParser()

# 3) Ask a question end-to-end.
question = "What do convolutions detect?"
top = retrieve(question, chunks, k=2)
context = "\n".join(c.page_content for c in top)

print("Retrieved context:\n", context, "\n")
print("Answer:", rag_chain.invoke({"context": context, "question": question}))

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved context:
 Convolutions detect local patterns like edges, corners, and textures.
Computer vision lets computers interpret images and video. 

Answer: <|system|>
Answer the question using ONLY the context. If unsure, say you don't know.<|end|>
<|user|>
Context:
Convolutions detect local patterns like edges, corners, and textures.
Computer vision lets computers interpret images and video.

Question: What do convolutions detect?<|end|>
<|assistant|>
Convolutions detect local patterns such as edges, corners, and textures.


**You just built a full LLM pipeline from scratch** — model, prompt, parser, chain, memory, a loader, a splitter, and retrieval — using only local, free tools.

### Recap
| Fundamentals | Components |
|---|---|
| **Models** — interchangeable engines | **Memory** — remembers the conversation |
| **Prompts** — templated instructions | **Document Loaders** — data → `Document` objects |
| **Output Parsers** — text → usable data | **Text Splitters** — chunk for retrieval |
| **Chains / LCEL** — `prompt \| model \| parser` | |

### Where to go next
- Swap `flan-t5-small` → `flan-t5-base` and compare answer quality.
- Replace the toy retriever with real **embeddings + a vector store** for semantic search.
- Add **summarization** memory so long chats stay within the context window.

### References (live sources, retrieved July 2026)
- LangChain — Overview & LCEL: https://docs.langchain.com/oss/python/langchain/overview
- LangChain — Short-term memory: https://docs.langchain.com/oss/python/langchain/short-term-memory
- LangChain — Retrieval & RAG: https://docs.langchain.com/oss/python/langchain/retrieval
- ChatHuggingFace integration: https://docs.langchain.com/oss/python/integrations/chat/huggingface
- `RunnableWithMessageHistory` reference: https://reference.langchain.com/python/langchain-core/runnables/history/RunnableWithMessageHistory
